In [1]:
import os, json, time, hmac, hashlib, requests
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool, StructuredTool
from langchain_core.messages import HumanMessage, ToolMessage
from pydantic import BaseModel, Field

load_dotenv()
BASE_URL = "https://jsonplaceholder.typicode.com"

In [2]:
llm = ChatOpenAI(model='gpt-4o-mini')

In [3]:
# HTTP : 클라이언트 -> 요청을 보냅니다 (request)
#        서버 -> 응답 (response)

In [4]:
# request : 랭체인 설명해줘
    
# Method : 뭘 할건지?  POST/GET/PUT/DELETE
# URL : 어디서?        www.naver.com , BASE_URL
# 헤더 : 니가 누군데?  {인증정보, chrome/mozila ...}
# 바디 : 뭘 보낼건지?  {'title' : '제목입니다', 'content' : '내용입니다'}

In [5]:
# ok
# 200 : 조회 성공
# 201 : created 성공

# fail
# 400 : 요청 형식 틀림
# 401 : unauthorized
# 403 : forbidden

# 500 internal server error


In [6]:
r = requests.get(f"{BASE_URL}/posts/1")

In [7]:
r.status_code

200

In [8]:
r.json()

{'userId': 1,
 'id': 1,
 'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit',
 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}

In [9]:
r = requests.post(f"{BASE_URL}/posts", 
             json = {'title' : '새 글', 'body' : '본문', 'userId' : 1})

In [10]:
r.status_code

201

In [11]:
r.json()

{'title': '새 글', 'body': '본문', 'userId': 1, 'id': 101}

In [12]:
r = requests.delete(f"{BASE_URL}/posts/1", timeout=5)

In [13]:
r.status_code

200

In [14]:
r.json()

{}

In [15]:
session = requests.Session()
session.headers.update({
    'User-Agent' : 'abcde/1.0',
    'Accept' : 'application/json'
})

In [16]:
r = session.get(f"{BASE_URL}/posts", params = {"userId":1})

In [17]:
len(r.json())

10

In [18]:
r2 = session.get(f"{BASE_URL}/users/1")
r2.json()['name'], r2.json()['email']

('Leanne Graham', 'Sincere@april.biz')

In [19]:
r = session.post(f"{BASE_URL}/posts", json = {'title' : '새 글2', 'body' : '본문2', 'userId' : 1})

In [20]:
# curl -X POST "https://jsonplaceholder.typicode.com/posts" \
#     -H 

In [21]:
r.json()

{'title': '새 글2', 'body': '본문2', 'userId': 1, 'id': 101}

In [22]:
r.ok

True

In [23]:
# https://jsonplaceholder.typicode.com/posts

In [24]:
def rank_users_by_posts(user_ids):
    s = requests.Session()
    session.headers.update({
        'User-Agent' : 'abcde/1.0',
        'Accept' : 'application/json'
    })
    
    counts = []
    for uid in user_ids:
        r = session.get(f"{BASE_URL}/posts", params = {"userId": uid})
        counts.append({"userId" : uid, "post_count" : len(r.json())})
        
    counts.sort(key=lambda x : x["post_count"], reverse=True)
    for i, item in enumerate(counts, 1):
        item['rank'] = i
    
    return counts

In [25]:
rank_users_by_posts([1,2,3,4,5])

[{'userId': 1, 'post_count': 10, 'rank': 1},
 {'userId': 2, 'post_count': 10, 'rank': 2},
 {'userId': 3, 'post_count': 10, 'rank': 3},
 {'userId': 4, 'post_count': 10, 'rank': 4},
 {'userId': 5, 'post_count': 10, 'rank': 5}]

In [26]:
# https://jsonplaceholder.typicode.com/users/1
@tool
def fetch_user(user_id):
    """지정된 ID의 사용자 정보를 조회합니다"""
    r = requests.get(f"{BASE_URL}/users/{user_id}")
    if not r.ok:
        return f"error: status {r.status_code}"
    data = r.json()
    return json.dumps({
        'name' : data['name'],
        'email' : data['email'],
        'phone' : data['phone'],
        'company' : data['company']['name']
    }, ensure_ascii=False)

@tool
def fetch_user_post(user_id):
    """지정된 ID의 사용자가 작성한 게시글을 조회합니다"""
    r = requests.get(f"{BASE_URL}/posts", params={'userId':user_id})
    if not r.ok:
        return f"error: status {r.status_code}"
    posts = r.json()
    return json.dumps([{'id' : p['id'], 'title' : p['title']} for p in posts], ensure_ascii=False)

In [27]:
llm_api = llm.bind_tools([fetch_user, fetch_user_post])
response = llm_api.invoke("1번 유저 정보와 그 사람이 쓴 게시글 갯수 알려줘")
print(f"호출된 도구 개수 : {len(response.tool_calls)}")

호출된 도구 개수 : 2


In [28]:
for tc in response.tool_calls:
    print(f"{tc['name']} : {tc['args']}")

fetch_user : {'user_id': 1}
fetch_user_post : {'user_id': 1}


In [29]:
tool_map = {'fetch_user' : fetch_user, 'fetch_user_post' : fetch_user_post}
messages = [HumanMessage(content = '1번 유저 정보와 그 사람이 쓴 게시글 갯수 알려줘')]
resp = llm_api.invoke(messages)
messages.append(resp)

for tc in resp.tool_calls:
    result = tool_map[tc['name']].invoke(tc['args'])
    messages.append(ToolMessage(content=result, tool_call_id = tc['id']))
    
final = llm_api.invoke(messages)

In [30]:
final

AIMessage(content='1번 유저의 정보는 다음과 같습니다:\n\n- **이름**: Leanne Graham\n- **이메일**: Sincere@april.biz\n- **전화번호**: 1-770-736-8031 x56442\n- **회사**: Romaguera-Crona\n\n이 유저가 쓴 게시글의 갯수는 총 **10개**입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 383, 'total_tokens': 469, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f957560a82', 'id': 'chatcmpl-DY8NAu61LigNm6H94ZgTAM0Tr7jLG', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dbf1a-f52c-7da3-88e1-d13cf87d1cb9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 383, 'output_tokens': 86, 'total_tokens': 469, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_t

In [31]:
print(final.content)

1번 유저의 정보는 다음과 같습니다:

- **이름**: Leanne Graham
- **이메일**: Sincere@april.biz
- **전화번호**: 1-770-736-8031 x56442
- **회사**: Romaguera-Crona

이 유저가 쓴 게시글의 갯수는 총 **10개**입니다.


In [33]:
# https://jsonplaceholder.typicode.com/todos/?userId=1

In [34]:
@tool
def fetch_todos(user_id) : 
    """지정된 사용자의 todo 목록 상위 5개를 조회합니다"""
    r = requests.get(f"{BASE_URL}/todos", params={'userID' : user_id})
    if not r.ok:
        return f"error : {r.status_code}"
    todos = r.json()[:5]
    return json.dumps(todos, ensure_ascii=False)

@tool
def count_complted_todos(user_id):
    """지정된 사용자의 완료된 todo 개수와 전체 개수를 계산합니다"""
    r = requests.get(f"{BASE_URL}/todos", params={'userID' : user_id})
    if not r.ok:
        return f"error : {r.status_code}"
    
    todos = r.json()
    completed = sum(1 for t in todos if t['completed'])
    total = len(todos)
    return json.dumps({
        'user_id' : user_id,
        'completed' : completed,
        'total' : total,
        'completion_rate' : round(completed/total * 100, 1) if total else 0,
        }, ensure_ascii=False)

In [35]:
todo_tools = [fetch_todos, count_complted_todos]
llm_todo = llm.bind_tools(todo_tools)
tool_map = {t.name:t for t in todo_tools}

messages = [HumanMessage(content = '1번 유저의 todo 완료율을 알려줘')]
resp = llm_todo.invoke(messages)
messages.append(resp)

print(f"tools : {[tc['name'] for tc in resp.tool_calls]}")

for tc in resp.tool_calls:
    result = tool_map[tc['name']].invoke(tc['args'])
    messages.append(ToolMessage(content=result, tool_call_id = tc['id']))
    
final = llm_todo.invoke(messages)

tools : ['count_complted_todos', 'fetch_todos']


In [36]:
print(final.content)

1번 유저의 todo 완료율은 45%입니다. 현재 완료된 todo는 90개, 전체 todo는 200개입니다. 또한, 1번 유저의 상위 5개 todo 목록은 다음과 같습니다:

1. **delectus aut autem** - 미완료
2. **quis ut nam facilis et officia qui** - 미완료
3. **fugiat veniam minus** - 미완료
4. **et porro tempora** - 완료
5. **laboriosam mollitia et enim quasi adipisci quia provident illum** - 미완료


In [ ]:
# https://jsonplaceholder.typicode.com/posts
# https://jsonplaceholder.typicode.com/posts/?_page=1&_limit=10

In [39]:
def fetch_all_posts(page_size = 10, max_pages=100):
    page = 1
    while page <= max_pages:
        r = requests.get(f"{BASE_URL}/posts", params={"_page" : page, "_limit" : page_size})
        items = r.json()
        if not items:
            break
        print(f" page {page} : {len(items)}")
        for item in items:
            yield item
        if len(items) < page_size:
            break
        
        page += 1
    

In [40]:
all_posts = list(fetch_all_posts(page_size=20))

 page 1 : 20
 page 2 : 20
 page 3 : 20
 page 4 : 20
 page 5 : 20


In [42]:
# all_posts

In [43]:
len(all_posts), all_posts[0]['title']

(100,
 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit')

In [44]:
def cursor_paginate_simulator(total_items=25, page_size=10):
    all_data =[{"id" : i, "name" : f"item-{i}"} for i in range(1, total_items+1)]
    
    def fetch_page(cursor =None, limit=page_size):
        start = int(cursor) if cursor else 0
        end = min(start + limit, total_items)
        items = all_data[start:end]
        next_cursor = str(end) if end <total_items else None
        return {'items' : items, 'next_cursor' : next_cursor}
    
    return fetch_page

fetch = cursor_paginate_simulator(total_items=25, page_size=10)

In [45]:
cursor = None
page = 1
all_items = []
while True:
    result = fetch(cursor=cursor) # result = {'items' : items, 'next_cursor' : next_cursor}
    print(f"page : {page} | {len(result['items'])}, next = {result['next_cursor']}")
    all_items.extend(result['items'])
    if not result['next_cursor']:
        break
    cursor = result['next_cursor']
    page += 1


page : 1 | 10, next = 10
page : 2 | 10, next = 20
page : 3 | 5, next = None


In [46]:
all_items

[{'id': 1, 'name': 'item-1'},
 {'id': 2, 'name': 'item-2'},
 {'id': 3, 'name': 'item-3'},
 {'id': 4, 'name': 'item-4'},
 {'id': 5, 'name': 'item-5'},
 {'id': 6, 'name': 'item-6'},
 {'id': 7, 'name': 'item-7'},
 {'id': 8, 'name': 'item-8'},
 {'id': 9, 'name': 'item-9'},
 {'id': 10, 'name': 'item-10'},
 {'id': 11, 'name': 'item-11'},
 {'id': 12, 'name': 'item-12'},
 {'id': 13, 'name': 'item-13'},
 {'id': 14, 'name': 'item-14'},
 {'id': 15, 'name': 'item-15'},
 {'id': 16, 'name': 'item-16'},
 {'id': 17, 'name': 'item-17'},
 {'id': 18, 'name': 'item-18'},
 {'id': 19, 'name': 'item-19'},
 {'id': 20, 'name': 'item-20'},
 {'id': 21, 'name': 'item-21'},
 {'id': 22, 'name': 'item-22'},
 {'id': 23, 'name': 'item-23'},
 {'id': 24, 'name': 'item-24'},
 {'id': 25, 'name': 'item-25'}]

In [ ]:
https://jsonplaceholder.typicode.com/todos

In [ ]:
def fetch_all_posts(page_size = 10, max_pages=100):
    page = 1
    while page <= max_pages:
        r = requests.get(f"{BASE_URL}/posts", params={"_page" : page, "_limit" : page_size})
        items = r.json()
        if not items:
            break
        print(f" page {page} : {len(items)}")
        for item in items:
            yield item
        if len(items) < page_size:
            break
        
        page += 1
    

In [47]:
# 1. completed = True 인 todo 만 가져옵니다
# 2. 20개가 모이면 중단
# 3. completed=True 인 todo list 20개를 리턴

def collect_completed_todos(max_items=20, page_size=10):
    collected = []
    page = 1
    while len(collected) < max_items:
        r = requests.get(f"{BASE_URL}/todos", params={"_page" : page, "_limit" : page_size})
        items = r.json()
        if not items:
            break
        completed = [t for t in items if t.get('completed')]
        collected.extend(completed[:max_items - len(collected)])
        print(f" page {page} : {len(items)}, collected : {len(collected)}")

        if len(items) < page_size:
            break
        page +=1
        
    
    return collected[:max_items]

result = collect_completed_todos(max_items=10, page_size=15)
result

 page 1 : 15, collected : 7
 page 2 : 15, collected : 10


[{'userId': 1, 'id': 4, 'title': 'et porro tempora', 'completed': True},
 {'userId': 1,
  'id': 8,
  'title': 'quo adipisci enim quam ut ab',
  'completed': True},
 {'userId': 1,
  'id': 10,
  'title': 'illo est ratione doloremque quia maiores aut',
  'completed': True},
 {'userId': 1,
  'id': 11,
  'title': 'vero rerum temporibus dolor',
  'completed': True},
 {'userId': 1,
  'id': 12,
  'title': 'ipsa repellendus fugit nisi',
  'completed': True},
 {'userId': 1,
  'id': 14,
  'title': 'repellendus sunt dolores architecto voluptatum',
  'completed': True},
 {'userId': 1,
  'id': 15,
  'title': 'ab voluptatum amet voluptas',
  'completed': True},
 {'userId': 1,
  'id': 16,
  'title': 'accusamus eos facilis sint et aut voluptatem',
  'completed': True},
 {'userId': 1,
  'id': 17,
  'title': 'quo laboriosam deleniti aut qui',
  'completed': True},
 {'userId': 1,
  'id': 19,
  'title': 'molestiae ipsa aut voluptatibus pariatur dolor nihil',
  'completed': True}]

In [ ]:
r ={
  "id": 1,
  "name": "Leanne Graham",
  "username": "Bret",
  "email": "Sincere@april.biz",
  "address": {
    "street": "Kulas Light",
    "suite": "Apt. 556",
    "city": "Gwenborough",
    "zipcode": "92998-3874",
    "geo": {
      "lat": "-37.3159",
      "lng": "81.1496"
    }
  },
  "phone": "1-770-736-8031 x56442",
  "website": "hildegard.org",
  "company": {
    "name": "Romaguera-Crona",
    "catchPhrase": "Multi-layered client-server neural-net",
    "bs": "harness real-time e-markets"
  }
}

In [ ]:
address.city : r['address']['city']

In [49]:
def get_by_path(data, path, default=None):
    current = data
    for key in path.split("."): # [address, city]
        if isinstance(current, dict) and key in current:
            current = current[key]
        else:
            return default
    return current

user_data = requests.get(f"{BASE_URL}/users/1").json()
get_by_path(user_data, 'name'), get_by_path(user_data, 'address.city')

('Leanne Graham', 'Gwenborough')

In [50]:
def get_by_path(data, path, default=None):
    current = data
    for key in path.split("."): # [address, city]
        if isinstance(current, dict) and key in current:
            current = current[key]
        else:
            return default
    return current
    

def users_to_markdown(user_ids : list, fields: list):
    rows = []
    for uid in user_ids:
        r = requests.get(f"{BASE_URL}/users/{uid}")   # https://jsonplaceholder.typicode.com/users/1
        if not r.ok:
            continue
        data = r.json()
        row = {f:get_by_path(data, f, "N/A") for f in fields}
        rows.append(row)
    
    header = "| " + " | ".join(fields) + " |"   
#     | name | email | company |
    sep = "|" + "|".join(["-" * (len(f) +2) for f in fields] ) + "|"
    body_lines = []
    for row in rows:
        line = "| " + " | ".join(str(row[f]) for f in fields) + " |"
        body_lines.append(line)
    
    return '\n'.join([header, sep] + body_lines)

table = users_to_markdown(user_ids = [1,2,3], fields = ['name', 'email', 'address.city'])

In [52]:
print(table)

| name | email | address.city |
|------|-------|--------------|
| Leanne Graham | Sincere@april.biz | Gwenborough |
| Ervin Howell | Shanna@melissa.tv | Wisokyburgh |
| Clementine Bauch | Nathan@yesenia.net | McKenziehaven |


In [ ]:
# https://jsonplaceholder.typicode.com/users/1
# https://jsonplaceholder.typicode.com/posts?userId=1
# https://jsonplaceholder.typicode.com/todos?userId=1
# https://jsonplaceholder.typicode.com/comments?userId=1

In [61]:
def build_user_profile(user_id):
    profile = {'user_id' : user_id}
    
    # basic info
    try:
        r = requests.get(f"{BASE_URL}/users/{user_id}")
        if r.ok:
            u = r.json()
            profile['name'] = u['name']
            profile['email'] = u['email']
            profile['city'] = u['address']['city']
    except:
        profile['user_error'] = 'error'
    
    # post
    try:
        r = requests.get(f"{BASE_URL}/posts", params = {'userId' : user_id})
        if r.ok:
            posts = r.json()
            profile['post_count'] = len(posts)
            profile['recent_posts'] = [p['title'][:40] for p in posts[:3]]
    except Exception as e:
        profile['post_error'] = str(e)
    
    # todo
    try:
        r = requests.get(f"{BASE_URL}/todos", params = {'userId' : user_id})
        if r.ok:
            todos = r.json()
            profile['todo_total'] = len(todos)
            profile['todo_completed'] = sum(1 for t in todos if t['completed'])
    except:
        profile['todo_error'] = 'error'
        
    return profile

In [62]:
build_user_profile(1)

{'user_id': 1,
 'name': 'Leanne Graham',
 'email': 'Sincere@april.biz',
 'city': 'Gwenborough',
 'post_count': 10,
 'recent_posts': ['sunt aut facere repellat provident occae',
  'qui est esse',
  'ea molestias quasi exercitationem repell'],
 'todo_total': 20,
 'todo_completed': 11}

In [63]:
def build_team_report(user_list):
    members = []
    for uid in user_list:
        try:
            u = requests.get(f"{BASE_URL}/users/{uid}")
            p = requests.get(f"{BASE_URL}/posts", params = {'userId' : uid})
            t = requests.get(f"{BASE_URL}/todos", params = {'userId' : uid})
            if not (u.ok and p.ok and t.ok):
                continue
            todos = t.json()
            members.append({
                'user_id' : uid,
                'name' : u.json()['name'],
                'post_count' : len(p.json())
            })
        except:
            continue
    
    total_posts = sum(m['post_count'] for m in members)
    top = max(members, key=lambda m: m['post_count']) if members else None
    
    return {
        'team_size' : len(members),
        'members' : members,
        'top_poster' : {'user_id' : top['user_id'], 'post_count' : top['post_count']} if top else None
    }



In [64]:
build_team_report([1,2,3,4,5])

{'team_size': 5,
 'members': [{'user_id': 1, 'name': 'Leanne Graham', 'post_count': 10},
  {'user_id': 2, 'name': 'Ervin Howell', 'post_count': 10},
  {'user_id': 3, 'name': 'Clementine Bauch', 'post_count': 10},
  {'user_id': 4, 'name': 'Patricia Lebsack', 'post_count': 10},
  {'user_id': 5, 'name': 'Chelsey Dietrich', 'post_count': 10}],
 'top_poster': {'user_id': 1, 'post_count': 10}}

In [66]:
@tool
def api_get_user(user_id):
    """사용자 기본 정보(이름, 이메일, 도시)를 조회합니다"""
    r = requests.get(f"{BASE_URL}/users/{user_id}")
    u = r.json()
    return json.dumps({'name' : u['name'], 'email' : u['email'], 'city' : u['address']['city']}, ensure_ascii=False)

@tool
def api_get_user_posts(user_id, limit=5):
    """사용자의 최근 게시글 제목을 조회합니다"""
    r = requests.get(f"{BASE_URL}/posts", params = {'userId' : user_id})
    posts = r.json()[:limit]
    return json.dumps([{'id' : p['id'], 'title' : p['title']} for p in posts], ensure_ascii=False)

@tool
def api_get_user_todos(user_id):
    """사용자의 TODO 완료율을 조회합니다"""
    r = requests.get(f"{BASE_URL}/todos", params = {'userId' : user_id})
    todos = r.json()
    done = sum(1 for t in todos if t['completed'])
    return json.dumps({'completed' : done, 'total' : len(todos), 'rate' : f'{done/len(todos)*100:.0f}%'}, ensure_ascii=False)

api_tools = [api_get_user, api_get_user_posts, api_get_user_todos]
tool_map = {t.name: t for t in api_tools}

print(f"registered tools : {[t.name for t in api_tools]}")

registered tools : ['api_get_user', 'api_get_user_posts', 'api_get_user_todos']


In [ ]:
'api_get_users'

In [68]:
# 오케스트레이터 
def orchestrate(query, max_turns= 5):
    llm_api = llm.bind_tools(api_tools)
    messages = [HumanMessage(content=query)]
    
    tool_call_count = 0
    MAX_TOOL_CALLS = 10
    
    for turn in range(max_turns):
        try:
            resp = llm_api.invoke(messages)
        except Exception as e:
            return f"llm call fail {e}"
        messages.append(resp)
        
        if not resp.tool_calls:
            return resp.content
        
        for tc in resp.tool_calls:
            name = tc['name']
            args = tc['args']
            
            if name not in tool_map:
                result = f"Error : unkown tool '{name}'"
            else:
                tool_call_count +=1
                if tool_call_count > MAX_TOOL_CALLS:
                    return f"Error : tool call limit exceeded"
                
                try:
                    result = tool_map[tc['name']].invoke(tc['args'])
                except:
                    result = f"Error "
            print(f" [Turn {turn+1}] {tc['name']} ({tc['args']}) -> {result[:80]}")
            messages.append(ToolMessage(content=result, tool_call_id = tc['id']))
    return '최대 턴 초과'

orchestrate('1번 유저의 정보와 최근 게시글 3개, todo 완료율을 알려줘')

 [Turn 1] api_get_user ({'user_id': 1}) -> {"name": "Leanne Graham", "email": "Sincere@april.biz", "city": "Gwenborough"}
 [Turn 1] api_get_user_posts ({'user_id': 1, 'limit': 3}) -> [{"id": 1, "title": "sunt aut facere repellat provident occaecati excepturi opti
 [Turn 1] api_get_user_todos ({'user_id': 1}) -> {"completed": 11, "total": 20, "rate": "55%"}


'1번 유저에 대한 정보는 다음과 같습니다:\n\n- **이름**: Leanne Graham\n- **이메일**: Sincere@april.biz\n- **도시**: Gwenborough\n\n최근 게시글 3개는 다음과 같습니다:\n\n1. **제목**: sunt aut facere repellat provident occaecati excepturi optio reprehenderit\n2. **제목**: qui est esse\n3. **제목**: ea molestias quasi exercitationem repellat qui ipsa sit aut\n\nTODO 완료율은 다음과 같습니다:\n\n- **완료된 항목**: 11개\n- **전체 항목**: 20개\n- **완료율**: 55%'

In [69]:
from langchain_core.tools import tool, StructuredTool, BaseTool

In [ ]:
# 인스턴스 변수, 캐시, ..

In [70]:
class APICallerTool(BaseTool):
    name : str = "api_caller"
    description : str = "REST API를 호출하여 데이터를 조회합니다"
    call_count : int = 0
    cache : dict = {}
        
    def _run(self, endpoint: str) -> str:
        # 캐시 확인
        if endpoint in self.cache:
            return f"[cache] {self.cache[endpoint]}"
    
        self.call_count += 1
        data = {
            "users/1" : {"name" : "abcd", 'email' : 'abcd@example.com'},
            "posts/1" : {'title' : '1st post', 'userId' : 1},
            'products/1' : {'name' : 'labtop', 'price' : 150000}
        }
        result = json.dumps(data.get(endpoint, {'error' : 'not found'}), ensure_ascii=False)
        self.cache[endpoint] = result
        return f"[API #{self.call_count}] {result}"

In [71]:
api_tool = APICallerTool()
api_tool.invoke({'endpoint' : 'users/1'})

'[API #1] {"name": "abcd", "email": "abcd@example.com"}'

In [72]:
api_tool.invoke({'endpoint' : 'posts/1'})

'[API #2] {"title": "1st post", "userId": 1}'

In [73]:
api_tool.invoke({'endpoint' : 'products/1'})

'[API #3] {"name": "labtop", "price": 150000}'

In [75]:
from typing import Type

In [76]:
class StockInput(BaseModel):
    ticker : str = Field(description = "주식 티커 (예: AAPL)")
    include_history : bool = Field(default=False, description = "과거 데이터 포함 여부")

class StockTool(BaseTool):
    name : str = "stock_lookup"
    description : str = "주식 정보를 조회합니다"
    args_schema : Type[BaseModel] = StockInput
    query_log : list = []
        
    def _run(self, ticker:str, include_history: bool=False) -> str:
        prices = {"AAPL" : 100.5, "GOOGL" : 102.1, "TSLA": 280.3}
        price = prices.get(ticker.upper())
        if not price:
            return f"{ticker} no data"
        
        self.query_log.append({"ticker" : ticker, "time" : time.time()})
        result = f"{ticker} : ${price}"
        if include_history:
            result += f" (30일 변동 : + {price*0.05})"
        return result

In [77]:
stock = StockTool()

In [78]:
stock.invoke({'ticker' : 'AAPL'})

'AAPL : $100.5'

In [80]:
stock.invoke({'ticker' : 'GOOGL', 'include_history' : True})

'GOOGL : $102.1 (30일 변동 : + 5.105)'

In [81]:
stock.query_log

[{'ticker': 'AAPL', 'time': 1777036020.4329085},
 {'ticker': 'GOOGL', 'time': 1777036047.752681}]

In [86]:
class CachedSearchTool(BaseTool):
    name : str = "cached_search"
    description : str = "검색 결과를 캐싱하는 툴입니다"
    cache : dict = {}
    api_call : int = 0
    
    def _run(self, query: str) -> str:
        if query in self.cache:
            return f"[cache] {self.cache[query]}"
        
        self.api_call += 1
        result = f"'{query}' 검색 결과 {self.api_call}건"
        
        self.cache[query] = result
        return f"[API] {result}"


In [87]:
search = CachedSearchTool()
print(search.invoke({'query' : '파이썬 기초'}))
print(search.invoke({'query' : '파이썬 기초'}))

[API] '파이썬 기초' 검색 결과 1건
[cache] '파이썬 기초' 검색 결과 1건
